### Importing the Necessary Libraries

In [ ]:
import torch, torchvision
from torchvision import models, datasets, transforms
import torch.nn as nn, torch.optim as optim
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix
import matplotlib.pyplot as plt
import numpy as np
import os

import torch

# Check if CUDA GPU is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# (Optional) if multiple GPUs are available
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))



Using device: cuda
GPU name: NVIDIA A100-SXM4-40GB


In [ ]:
import kagglehub, os

# Download dataset
path = kagglehub.dataset_download("birdy654/cifake-real-and-ai-generated-synthetic-images")
print("Dataset path:", path)

# Check contents
!ls -R {path}


Streaming output truncated to the last 5000 lines.
'0750 (10).jpg'  '2000 (10).jpg'  '3250 (10).jpg'  '4500 (10).jpg'
'0750 (2).jpg'	 '2000 (2).jpg'   '3250 (2).jpg'   '4500 (2).jpg'
'0750 (3).jpg'	 '2000 (3).jpg'   '3250 (3).jpg'   '4500 (3).jpg'
'0750 (4).jpg'	 '2000 (4).jpg'   '3250 (4).jpg'   '4500 (4).jpg'
'0750 (5).jpg'	 '2000 (5).jpg'   '3250 (5).jpg'   '4500 (5).jpg'
'0750 (6).jpg'	 '2000 (6).jpg'   '3250 (6).jpg'   '4500 (6).jpg'
'0750 (7).jpg'	 '2000 (7).jpg'   '3250 (7).jpg'   '4500 (7).jpg'
'0750 (8).jpg'	 '2000 (8).jpg'   '3250 (8).jpg'   '4500 (8).jpg'
'0750 (9).jpg'	 '2000 (9).jpg'   '3250 (9).jpg'   '4500 (9).jpg'
 0750.jpg	  2000.jpg	   3250.jpg	    4500.jpg
'0751 (10).jpg'  '2001 (10).jpg'  '3251 (10).jpg'  '4501 (10).jpg'
'0751 (2).jpg'	 '2001 (2).jpg'   '3251 (2).jpg'   '4501 (2).jpg'
'0751 (3).jpg'	 '2001 (3).jpg'   '3251 (3).jpg'   '4501 (3).jpg'
'0751 (4).jpg'	 '2001 (4).jpg'   '3251 (4).jpg'   '4501 (4).jpg'
'0751 (5).jpg'	 '2001 (5).jpg'   '3251 (5).jpg'   '450

In [ ]:
import os, shutil, random

# Current dataset path
data_dir = path

# Where we want to organize
base_dir = "/content/data"
train_dir = os.path.join(base_dir, "train")
test_dir  = os.path.join(base_dir, "test")

# Create structure
for split in ["train", "test"]:
    for cls in ["FAKE", "REAL"]:
        os.makedirs(os.path.join(base_dir, split, cls), exist_ok=True)

# Collect all images
all_imgs = [f for f in os.listdir(data_dir) if f.endswith(".jpg")]

# CIFAKE names encode labels: fake_123.jpg or real_456.jpg
fake_imgs = [f for f in all_imgs if "fake" in f.lower()]
real_imgs = [f for f in all_imgs if "real" in f.lower()]

# Train/Test split (80/20)
random.seed(42)
def split_and_copy(img_list, cls_name):
    random.shuffle(img_list)
    split = int(0.8 * len(img_list))
    train, test = img_list[:split], img_list[split:]
    for f in train:
        shutil.copy(os.path.join(data_dir, f), os.path.join(train_dir, cls_name, f))
    for f in test:
        shutil.copy(os.path.join(data_dir, f), os.path.join(test_dir, cls_name, f))

split_and_copy(fake_imgs, "FAKE")
split_and_copy(real_imgs, "REAL")

print("Organized dataset at:", base_dir)
!tree {base_dir} -L 2


Organized dataset at: /content/data
/bin/bash: line 1: tree: command not found


In [ ]:
# --- CIFAKE auto-prepare for Colab ---
import os, glob, shutil, random, re
from pathlib import Path

# 1) If you used kagglehub earlier, you likely have this variable:
try:
    path  # noqa
except NameError:
    # fallback: point to where you downloaded; change if different
    path = "/root/.cache/kagglehub/datasets/birdy654/cifake-real-and-ai-generated-synthetic-images"
print("kagglehub base:", path)

def has_cifake_structure(root: str) -> bool:
    return (
        os.path.isdir(os.path.join(root, "train", "FAKE")) and
        os.path.isdir(os.path.join(root, "train", "REAL")) and
        os.path.isdir(os.path.join(root, "test",  "FAKE")) and
        os.path.isdir(os.path.join(root, "test",  "REAL"))
    )

def find_cifake_root(start: str):
    # search for a path that already has train/test/FAKE/REAL
    cand = None
    for p in glob.glob(os.path.join(start, "**"), recursive=True):
        if os.path.isdir(p) and has_cifake_structure(p):
            cand = p
            break
    return cand

# 2) Try to find an existing proper root inside kagglehub cache
root = find_cifake_root(path)

# 3) If not found, we will organize a new root at /content/data
dst_root = "/content/data"
if root is None:
    # create skeleton
    for split in ["train", "test"]:
        for cls in ["FAKE", "REAL"]:
            os.makedirs(os.path.join(dst_root, split, cls), exist_ok=True)

    # Case A: dataset already has separate class folders somewhere (e.g., 'fake' and 'real').
    # Try to locate any folder names that look like class names and copy them.
    class_dirs = []
    for cls in ["fake", "FAKE", "real", "REAL"]:
        class_dirs += glob.glob(os.path.join(path, "**", cls), recursive=True)

    copied = False
    if class_dirs:
        # Copy all images from any found class dirs into train/test (80/20)
        random.seed(42)
        for cls_path in class_dirs:
            cls_name = "FAKE" if re.search(r"fake", cls_path, re.IGNORECASE) else "REAL"
            imgs = []
            for ext in ("*.jpg","*.jpeg","*.png","*.bmp","*.webp","*.tif","*.tiff"):
                imgs += glob.glob(os.path.join(cls_path, ext))
            if not imgs:
                continue
            random.shuffle(imgs)
            k = int(0.8 * len(imgs))
            for f in imgs[:k]:
                shutil.copy2(f, os.path.join(dst_root, "train", cls_name, os.path.basename(f)))
            for f in imgs[k:]:
                shutil.copy2(f, os.path.join(dst_root, "test", cls_name, os.path.basename(f)))
            copied = True

    # Case B: flat folder with mixed images (filenames might encode the label)
    # Some CIFAKE dumps include 'real'/'fake' in file names; if so, split by name.
    if not copied:
        # collect all image files under path
        all_imgs = []
        for ext in ("*.jpg","*.jpeg","*.png","*.bmp","*.webp","*.tif","*.tiff"):
            all_imgs += glob.glob(os.path.join(path, "**", ext), recursive=True)
        fake_imgs = [f for f in all_imgs if re.search(r"\bfake\b", os.path.basename(f), re.IGNORECASE)]
        real_imgs = [f for f in all_imgs if re.search(r"\breal\b", os.path.basename(f), re.IGNORECASE)]

        if fake_imgs and real_imgs:
            random.seed(42)
            for cls_name, imgs in [("FAKE", fake_imgs), ("REAL", real_imgs)]:
                random.shuffle(imgs)
                k = int(0.8 * len(imgs))
                for f in imgs[:k]:
                    shutil.copy2(f, os.path.join(dst_root, "train", cls_name, os.path.basename(f)))
                for f in imgs[k:]:
                    shutil.copy2(f, os.path.join(dst_root, "test", cls_name, os.path.basename(f)))
            copied = True

    if not copied:
        # Last resort: suggest downloading a version that contains cifake/train/test/...
        raise RuntimeError(
            "Could not infer class folders or labels from this kagglehub download.\n"
            "Either use a CIFAKE variant that includes 'train/test/FAKE/REAL', or provide a labels CSV."
        )

    root = dst_root

print("Using CIFAKE root:", root)
# Show a quick tree (two levels)
!find {root} -maxdepth 2 -type d -print
# Count a few files to ensure non-empty
import glob as _glob
print("train/FAKE:", len(_glob.glob(os.path.join(root, "train", "FAKE", "*"))))
print("train/REAL:", len(_glob.glob(os.path.join(root, "train", "REAL", "*"))))
print("test/FAKE :", len(_glob.glob(os.path.join(root, "test",  "FAKE", "*"))))
print("test/REAL :", len(_glob.glob(os.path.join(root, "test",  "REAL", "*"))))

# Expose ROOT_DIR for your loader cell
ROOT_DIR = root


kagglehub base: /root/.cache/kagglehub/datasets/birdy654/cifake-real-and-ai-generated-synthetic-images/versions/3
Using CIFAKE root: /root/.cache/kagglehub/datasets/birdy654/cifake-real-and-ai-generated-synthetic-images/versions/3/
/root/.cache/kagglehub/datasets/birdy654/cifake-real-and-ai-generated-synthetic-images/versions/3/
/root/.cache/kagglehub/datasets/birdy654/cifake-real-and-ai-generated-synthetic-images/versions/3/test
/root/.cache/kagglehub/datasets/birdy654/cifake-real-and-ai-generated-synthetic-images/versions/3/test/REAL
/root/.cache/kagglehub/datasets/birdy654/cifake-real-and-ai-generated-synthetic-images/versions/3/test/FAKE
/root/.cache/kagglehub/datasets/birdy654/cifake-real-and-ai-generated-synthetic-images/versions/3/train
/root/.cache/kagglehub/datasets/birdy654/cifake-real-and-ai-generated-synthetic-images/versions/3/train/REAL
/root/.cache/kagglehub/datasets/birdy654/cifake-real-and-ai-generated-synthetic-images/versions/3/train/FAKE
train/FAKE: 50000
train/REAL

In [ ]:
# ==== CIFAKE loaders (Colab, CUDA) ====
import os, math, torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset

assert 'ROOT_DIR' in globals(), "Run the prep cell first so ROOT_DIR is defined."
print("ROOT_DIR:", ROOT_DIR)

IMG_SIZE   = 224
BATCH      = 128           # A100 can handle this; reduce if OOM
FRACTION   = 0.10          # use 10% for faster runs; set 1.0 for full data
VAL_RATIO  = 0.20          # make a val split from TRAIN (post-fraction)
SEED       = 42
NUM_WORKERS = 4

g = torch.Generator().manual_seed(SEED)

tf_train = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])
tf_eval = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

def stratified_split_indices(samples, num_classes, val_ratio, fraction, generator):
    per_class = {c: [] for c in range(num_classes)}
    for idx, (_, cls) in enumerate(samples):
        per_class[cls].append(idx)

    train_keep, val_keep = [], []
    for c, idxs in per_class.items():
        if not idxs: continue
        perm = torch.randperm(len(idxs), generator=generator).tolist()
        idxs = [idxs[i] for i in perm]
        k_frac = max(1, math.floor(len(idxs) * fraction))  # take only a fraction
        idxs = idxs[:k_frac]
        k_val = max(1, math.floor(len(idxs) * val_ratio))  # carve out val
        val_keep.extend(idxs[:k_val])
        train_keep.extend(idxs[k_val:])

    if train_keep:
        train_keep = [train_keep[i] for i in torch.randperm(len(train_keep), generator=generator).tolist()]
    if val_keep:
        val_keep   = [val_keep[i]   for i in torch.randperm(len(val_keep),   generator=generator).tolist()]
    return train_keep, val_keep

def stratified_subset_indices(samples, num_classes, fraction, generator):
    per_class = {c: [] for c in range(num_classes)}
    for idx, (_, cls) in enumerate(samples):
        per_class[cls].append(idx)
    keep = []
    for c, idxs in per_class.items():
        perm = torch.randperm(len(idxs), generator=generator).tolist()
        k = max(1, math.floor(len(idxs) * fraction))
        keep.extend([idxs[i] for i in perm[:k]])
    if keep:
        keep = [keep[i] for i in torch.randperm(len(keep), generator=generator).tolist()]
    return keep

# build datasets
train_full = datasets.ImageFolder(os.path.join(ROOT_DIR, "train"), transform=tf_train)
test_full  = datasets.ImageFolder(os.path.join(ROOT_DIR, "test"),  transform=tf_eval)

train_idx, val_idx = stratified_split_indices(
    train_full.samples, num_classes=len(train_full.classes),
    val_ratio=VAL_RATIO, fraction=FRACTION, generator=g
)
test_idx = stratified_subset_indices(test_full.samples, len(test_full.classes), FRACTION, g)

# wrap subsets (val uses eval transforms)
val_full = datasets.ImageFolder(os.path.join(ROOT_DIR, "train"), transform=tf_eval)
train_ds = Subset(train_full, train_idx)
val_ds   = Subset(val_full,   val_idx)
test_ds  = Subset(test_full,  test_idx)

train_dl = DataLoader(train_ds, batch_size=BATCH, shuffle=True,  num_workers=NUM_WORKERS, pin_memory=True)
val_dl   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
test_dl  = DataLoader(test_ds,  batch_size=BATCH, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

print("classes:", train_full.classes, "(class_to_idx:", train_full.class_to_idx, ")")
print(f"FULL train:{len(train_full)}  FULL test:{len(test_full)}")
print(f"SUBSET train:{len(train_ds)}  val:{len(val_ds)}  test:{len(test_ds)}")


ROOT_DIR: /root/.cache/kagglehub/datasets/birdy654/cifake-real-and-ai-generated-synthetic-images/versions/3/
classes: ['FAKE', 'REAL'] (class_to_idx: {'FAKE': 0, 'REAL': 1} )
FULL train:100000  FULL test:20000
SUBSET train:8000  val:2000  test:2000


In [ ]:
# ==== ResNet-50 train/val (CUDA + AMP) ====
import torch, torch.nn as nn, torch.optim as optim
from torchvision import models
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
model.fc = nn.Linear(model.fc.in_features, 2)
model.to(device)

crit = nn.CrossEntropyLoss(label_smoothing=0.1)
opt  = optim.AdamW(model.parameters(), lr=3e-4, weight_decay=5e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=5)  # small run
scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

EPOCHS = 5
pos_idx = train_full.class_to_idx.get("FAKE", 1)  # for AUROC

for epoch in range(1, EPOCHS+1):
    # --- train ---
    model.train()
    running = 0.0
    for xb, yb in train_dl:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad()
        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
            logits = model(xb)
            loss   = crit(logits, yb)
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
        running += loss.item() * xb.size(0)
    train_loss = running / max(1, len(train_dl.dataset))
    sched.step()

    # --- val ---
    model.eval()
    y_true, y_pred, y_prob = [], [], []
    with torch.no_grad(), torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
        for xb, yb in val_dl:
            xb = xb.to(device)
            logits = model(xb)
            probs  = torch.softmax(logits, dim=1)
            y_pred.extend(logits.argmax(1).cpu().numpy())
            y_true.extend(yb.numpy())
            y_prob.extend(probs[:, pos_idx].cpu().numpy())

    acc = accuracy_score(y_true, y_pred)
    f1  = f1_score(y_true, y_pred)
    try:
        auroc = roc_auc_score(y_true, np.array(y_prob), pos_label=pos_idx)
    except Exception:
        auroc = float("nan")

    print(f"Epoch {epoch:02d} | train loss {train_loss:.4f} | val Acc {acc:.3f} F1 {f1:.3f} AUROC {auroc:.3f}")


Using device: cuda | GPU: NVIDIA A100-SXM4-40GB
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 159MB/s]
/tmp/ipython-input-1154356196.py:17: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())
/tmp/ipython-input-1154356196.py:29: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
/tmp/ipython-input-1154356196.py:42: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):


Epoch 01 | train loss 0.3847 | val Acc 0.885 F1 0.872 AUROC nan


/tmp/ipython-input-1154356196.py:29: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
/tmp/ipython-input-1154356196.py:42: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):


Epoch 02 | train loss 0.2985 | val Acc 0.901 F1 0.893 AUROC nan


...

Epoch 03 | train loss 0.2639 | val Acc 0.951 F1 0.951 AUROC nan


/tmp/ipython-input-1154356196.py:29: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
/tmp/ipython-input-1154356196.py:42: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):


Epoch 04 | train loss 0.2432 | val Acc 0.954 F1 0.954 AUROC nan


/tmp/ipython-input-1154356196.py:29: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
/tmp/ipython-input-1154356196.py:42: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):


Epoch 05 | train loss 0.2261 | val Acc 0.958 F1 0.957 AUROC nan


In [ ]:
# --- Install and import CLIP (OpenCLIP) ---
!pip -q install open-clip-torch

import torch, numpy as np
import open_clip
from torchvision import transforms
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# Load a CLIP ViT-B/16 model + its preprocess
model_name, pretrained = "ViT-B-16", "openai"
clip_model, _, clip_preprocess = open_clip.create_model_and_transforms(model_name, pretrained=pretrained, device=device)
tokenizer = open_clip.get_tokenizer(model_name)

# Replace val/test transforms with CLIP's preprocess-sized pipeline (keep your train_dl for training tasks)
# We'll make fresh loaders that only differ by transform
from torch.utils.data import DataLoader
from torchvision import datasets
import os

assert 'ROOT_DIR' in globals()
clip_val = datasets.ImageFolder(os.path.join(ROOT_DIR, "train"), transform=clip_preprocess)  # we'll index val_idx
clip_test = datasets.ImageFolder(os.path.join(ROOT_DIR, "test"),  transform=clip_preprocess)

# Subset to your existing val/test indices so it's apples-to-apples
from torch.utils.data import Subset
val_clip_ds  = Subset(clip_val,  val_ds.indices)   # reuse val split made earlier
test_clip_ds = Subset(clip_test, test_ds.indices)

val_clip_dl  = DataLoader(val_clip_ds,  batch_size=128, shuffle=False, num_workers=4, pin_memory=True)
test_clip_dl = DataLoader(test_clip_ds, batch_size=128, shuffle=False, num_workers=4, pin_memory=True)

# ---- Prompt templates & classnames (prompt ensembling) ----
classnames = ["FAKE", "REAL"]
# Keep wording neutral; you can try many variants
templates = [
    "a photo that is AI-generated",
    "an AI-generated image",
    "a synthetic image made by an AI",
    "a real camera photograph",
    "an authentic human-captured photo",
    "a natural photo taken by a person",
]

def build_text_embeds(classnames, templates):
    all_text_embeds = []
    for cname in classnames:
        prompts = [t.replace("photo", cname.lower()+" photo") if ("FAKE" in cname or "REAL" in cname) and "photo" in t else t for t in templates]
        prompts = [p.replace("AI-generated", "AI-generated") for p in prompts]  # no-op; placeholder for edits
        tokens = tokenizer(prompts).to(device)
        with torch.no_grad(), torch.amp.autocast("cuda", enabled=torch.cuda.is_available()):
            text_features = clip_model.encode_text(tokens)
            text_features /= text_features.norm(dim=-1, keepdim=True)
        all_text_embeds.append(text_features)  # [num_templates, d]
    # stack: [num_classes, num_templates, d]
    return all_text_embeds

text_embeds = build_text_embeds(classnames, templates)

@torch.no_grad()
def clip_zeroshot_eval(dloader, name="VAL"):
    y_true, y_pred, y_prob = [], [], []
    pos_idx = train_full.class_to_idx.get("FAKE", 1)

    for xb, yb in dloader:
        xb = xb.to(device)
        with torch.amp.autocast("cuda", enabled=torch.cuda.is_available()):
            img = clip_model.encode_image(xb)
            img = img / img.norm(dim=-1, keepdim=True)  # [B, d]

            # similarity with each class' prompts, then average over templates
            # text_embeds[c]: [T, d]  -> sim: [B, T]
            sims = []
            for c in range(len(classnames)):
                te = text_embeds[c]
                sim = (img @ te.T) * 100.0
                sims.append(sim.mean(dim=1, keepdim=True))  # average templates
            logits = torch.cat(sims, dim=1)               # [B, C]
            probs  = torch.softmax(logits, dim=1).float()

        preds = probs.argmax(1).cpu().numpy()
        y_pred.extend(preds)
        y_true.extend(yb.numpy())
        y_prob.extend(probs[:, pos_idx].cpu().numpy())

    acc = accuracy_score(y_true, y_pred)
    f1  = f1_score(y_true, y_pred)
    try:
        auroc = roc_auc_score(y_true, np.array(y_prob), pos_label=pos_idx)
    except Exception:
        auroc = float("nan")
    cm = confusion_matrix(y_true, y_pred)
    print(f"[CLIP zero-shot] {name} → Acc {acc:.3f} | F1 {f1:.3f} | AUROC {auroc:.3f}\nCM:\n{cm}\n")
    return acc, f1, auroc

clip_zeroshot_val  = clip_zeroshot_eval(val_clip_dl,  "VAL")
clip_zeroshot_test = clip_zeroshot_eval(test_clip_dl, "TEST")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 25.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.5 MB/s eta 0:00:00
Device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


open_clip_model.safetensors:   0%|          | 0.00/599M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/open_clip/factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(


[CLIP zero-shot] VAL → Acc 0.468 | F1 0.496 | AUROC nan
CM:
[[411 589]
 [476 524]]

[CLIP zero-shot] TEST → Acc 0.460 | F1 0.496 | AUROC nan
CM:
[[387 613]
 [468 532]]



In [ ]:
# --- CLIP linear probe: freeze encoder, train a small head ---
import torch.nn as nn, torch.optim as optim
from torch.utils.data import DataLoader

# Build dataloaders that output CLIP image embeddings on the fly (for speed you could precompute, but this is fine on A100)
class ClipFeatDataset(torch.utils.data.Dataset):
    def __init__(self, subset, preprocess, model, device):
        self.subset = subset
        self.prep = preprocess
        self.model = model
        self.device = device
    def __len__(self): return len(self.subset)
    def __getitem__(self, i):
        img_path, label = self.subset.dataset.samples[self.subset.indices[i]]
        img = self.subset.dataset.loader(img_path)
        img = self.prep(img)
        with torch.no_grad(), torch.amp.autocast("cuda", enabled=torch.cuda.is_available()):
            feat = self.model.encode_image(img.unsqueeze(0).to(self.device))
            feat = feat / feat.norm(dim=-1, keepdim=True)
        return feat.squeeze(0).cpu(), label

clip_model.eval()  # freeze
train_feat_ds = ClipFeatDataset(Subset(clip_val, train_ds.indices), clip_preprocess, clip_model, device)
val_feat_ds   = ClipFeatDataset(val_clip_ds,                               clip_preprocess, clip_model, device)

train_feat_dl = DataLoader(train_feat_ds, batch_size=256, shuffle=True,
                           num_workers=0, pin_memory=True)
val_feat_dl   = DataLoader(val_feat_ds,   batch_size=256, shuffle=False,
                           num_workers=0, pin_memory=True)

# small linear head
d = clip_model.visual.output_dim
head = nn.Linear(d, 2).to(device)
opt  = optim.AdamW(head.parameters(), lr=1e-3, weight_decay=1e-4)
crit = nn.CrossEntropyLoss(label_smoothing=0.1)

EPOCHS = 5
pos_idx = train_full.class_to_idx.get("FAKE", 1)

for ep in range(1, EPOCHS+1):
    head.train(); run = 0.0
    for xb, yb in train_feat_dl:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad()
        logits = head(xb)
        loss   = crit(logits, yb)
        loss.backward(); opt.step()
        run += loss.item() * xb.size(0)
    tr_loss = run / max(1, len(train_feat_dl.dataset))

    head.eval(); y_true, y_pred, y_prob = [], [], []
    with torch.no_grad():
        for xb, yb in val_feat_dl:
            xb = xb.to(device)
            logits = head(xb)
            probs  = torch.softmax(logits, dim=1)
            y_pred.extend(logits.argmax(1).cpu().numpy())
            y_true.extend(yb.numpy())
            y_prob.extend(probs[:, pos_idx].cpu().numpy())
    acc = accuracy_score(y_true, y_pred); f1 = f1_score(y_true, y_pred)
    try: auroc = roc_auc_score(y_true, np.array(y_prob), pos_label=pos_idx)
    except: auroc = float("nan")
    print(f"[CLIP linear probe] Ep{ep:02d} | train {tr_loss:.4f} | VAL Acc {acc:.3f} F1 {f1:.3f} AUROC {auroc:.3f}")


[CLIP linear probe] Ep01 | train 0.6827 | VAL Acc 0.829 F1 0.839 AUROC nan
[CLIP linear probe] Ep02 | train 0.6628 | VAL Acc 0.842 F1 0.831 AUROC nan
[CLIP linear probe] Ep03 | train 0.6447 | VAL Acc 0.848 F1 0.843 AUROC nan
[CLIP linear probe] Ep04 | train 0.6279 | VAL Acc 0.849 F1 0.847 AUROC nan
[CLIP linear probe] Ep05 | train 0.6121 | VAL Acc 0.850 F1 0.845 AUROC nan


# CLIP FINE TUNING

In [ ]:
!pip -q install open-clip-torch

import os, torch, torch.nn as nn, torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix
import open_clip, math

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


Device: cuda | GPU: NVIDIA A100-SXM4-40GB


In [ ]:
# Load CLIP ViT-B/16 and its transforms
# NOTE: open_clip returns (model, preprocess_train, preprocess_val)
model_name, pretrained = "ViT-B-16", "openai"
clip_model, preprocess_train, preprocess_val = open_clip.create_model_and_transforms(
    model_name, pretrained=pretrained, device=device
)
clip_model.eval()  # we'll switch to train() later

# Build CLIP-style datasets that respect your existing indices
clip_train_full = datasets.ImageFolder(os.path.join(ROOT_DIR, "train"), transform=preprocess_train)
clip_val_full   = datasets.ImageFolder(os.path.join(ROOT_DIR, "train"), transform=preprocess_val)
clip_test_full  = datasets.ImageFolder(os.path.join(ROOT_DIR, "test"),  transform=preprocess_val)

train_clip_ds = Subset(clip_train_full, train_ds.indices)   # reuse the same split
val_clip_ds   = Subset(clip_val_full,   val_ds.indices)
test_clip_ds  = Subset(clip_test_full,  test_ds.indices)

BATCH = 128
NUM_WORKERS = 4
train_clip_dl = DataLoader(train_clip_ds, batch_size=BATCH, shuffle=True,  num_workers=NUM_WORKERS, pin_memory=True)
val_clip_dl   = DataLoader(val_clip_ds,   batch_size=BATCH, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
test_clip_dl  = DataLoader(test_clip_ds,  batch_size=BATCH, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

print("CLIP output dim:", clip_model.visual.output_dim)


/usr/local/lib/python3.12/dist-packages/open_clip/factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(


CLIP output dim: 512


In [ ]:
# Small head on top of CLIP image features
head = nn.Linear(clip_model.visual.output_dim, 2).to(device)

# Fine-tune settings
BACKBONE_LR = 1e-5     # small LR for the CLIP backbone
HEAD_LR     = 1e-3     # larger LR for the new head
WEIGHT_DECAY = 5e-4
EPOCHS = 10
CLIP_GRAD_NORM = 1.0
PATIENCE = 3           # early stopping on val F1

# Enable full fine-tune (unfreeze backbone)
for p in clip_model.parameters():
    p.requires_grad = True

# Param groups: smaller LR for backbone, bigger for head
opt = optim.AdamW(
    [
        {"params": [p for n,p in clip_model.named_parameters() if p.requires_grad], "lr": BACKBONE_LR},
        {"params": head.parameters(), "lr": HEAD_LR},
    ],
    weight_decay=WEIGHT_DECAY
)

# Cosine scheduler over total epochs
sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
scaler = torch.amp.GradScaler(enabled=torch.cuda.is_available())
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

pos_idx = train_full.class_to_idx.get("FAKE", 1)  # for AUROC
best_f1, best_state, patience = -1.0, None, PATIENCE


In [ ]:
def run_epoch_train():
    clip_model.train(); head.train()
    total = 0.0
    for xb, yb in train_clip_dl:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda", enabled=torch.cuda.is_available()):
            feats = clip_model.encode_image(xb)                    # [B, d]
            feats = F.normalize(feats, dim=-1)
            logits = head(feats)                                   # [B, 2]
            loss = criterion(logits, yb)
        scaler.scale(loss).backward()
        # Gradient clipping helps stability when fine-tuning transformers
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(list(clip_model.parameters()) + list(head.parameters()), CLIP_GRAD_NORM)
        scaler.step(opt); scaler.update()
        total += loss.item() * xb.size(0)
    return total / max(1, len(train_clip_dl.dataset))

@torch.no_grad()
def evaluate(dloader, name="VAL"):
    clip_model.eval(); head.eval()
    y_true, y_pred, y_prob = [], [], []
    for xb, yb in dloader:
        xb = xb.to(device)
        with torch.amp.autocast("cuda", enabled=torch.cuda.is_available()):
            feats = clip_model.encode_image(xb)
            feats = F.normalize(feats, dim=-1)
            logits = head(feats)
            probs  = torch.softmax(logits, dim=1)
        y_pred.extend(probs.argmax(1).cpu().numpy())
        y_true.extend(yb.numpy())
        y_prob.extend(probs[:, pos_idx].cpu().numpy())
    acc = accuracy_score(y_true, y_pred)
    f1  = f1_score(y_true, y_pred)
    try: auroc = roc_auc_score(y_true, np.array(y_prob), pos_label=pos_idx)
    except: auroc = float("nan")
    cm = confusion_matrix(y_true, y_pred)
    print(f"{name} → Acc {acc:.3f} | F1 {f1:.3f} | AUROC {auroc:.3f}")
    return acc, f1, auroc, cm

for ep in range(1, EPOCHS+1):
    tr_loss = run_epoch_train()
    acc, f1, auroc, _ = evaluate(val_clip_dl, "VAL")
    sched.step()
    print(f"Epoch {ep:02d} | train {tr_loss:.4f} | VAL Acc {acc:.3f} F1 {f1:.3f} AUROC {auroc:.3f}")

    # Early stopping on best val F1
    if f1 > best_f1:
        best_f1 = f1
        patience = PATIENCE
        best_state = {
            "clip": {k: v.detach().cpu() for k, v in clip_model.state_dict().items()},
            "head": head.state_dict(),
        }
    else:
        patience -= 1
        if patience == 0:
            print(f"Early stopping (best F1={best_f1:.3f}).")
            break


VAL → Acc 0.920 | F1 0.924 | AUROC nan
Epoch 01 | train 0.4661 | VAL Acc 0.920 F1 0.924 AUROC nan
VAL → Acc 0.917 | F1 0.911 | AUROC nan
Epoch 02 | train 0.3182 | VAL Acc 0.917 F1 0.911 AUROC nan
VAL → Acc 0.953 | F1 0.952 | AUROC nan
Epoch 03 | train 0.2591 | VAL Acc 0.953 F1 0.952 AUROC nan
VAL → Acc 0.967 | F1 0.967 | AUROC nan
Epoch 04 | train 0.2361 | VAL Acc 0.967 F1 0.967 AUROC nan
VAL → Acc 0.969 | F1 0.969 | AUROC nan
Epoch 05 | train 0.2234 | VAL Acc 0.969 F1 0.969 AUROC nan
VAL → Acc 0.954 | F1 0.952 | AUROC nan
Epoch 06 | train 0.2133 | VAL Acc 0.954 F1 0.952 AUROC nan
VAL → Acc 0.966 | F1 0.966 | AUROC nan
Epoch 07 | train 0.2072 | VAL Acc 0.966 F1 0.966 AUROC nan
VAL → Acc 0.968 | F1 0.968 | AUROC nan
Epoch 08 | train 0.2029 | VAL Acc 0.968 F1 0.968 AUROC nan
Early stopping (best F1=0.969).


In [ ]:
# Restore best
if best_state is not None:
    clip_model.load_state_dict({k: v.to(device) for k, v in best_state["clip"].items()})
    head.load_state_dict(best_state["head"])

# Final test metrics
acc_t, f1_t, auroc_t, cm_t = evaluate(test_clip_dl, "TEST")
print("Confusion matrix (TEST):\n", cm_t)


TEST → Acc 0.978 | F1 0.977 | AUROC nan
Confusion matrix (TEST):
 [[983  17]
 [ 28 972]]
